In [1]:
import pandas as pd
from scipy.stats import shapiro
import numpy as np
from scipy.stats import ttest_rel
import itertools
import matplotlib.pyplot as plt
from matplotlib.table import Table

# Paired T-Test

In [2]:
# Example data: columns = folds, rows = training strategies
# loss
data = np.array([
    [0.570404	,0.612975	,0.589749	,0.683906	,0.577115],  # img
    [0.74194	,0.724174	,0.668706	,0.677458	,0.651402], # tab
    [0.930846	,0.970325	,0.888275	,0.944269	,0.960258], # end
    [0.940913	,0.924661	,0.898079	,0.904264	,0.957165], # hyb ae
    [0.931373	,0.908409	,0.924135	,0.907619	,0.944006], # hyb single
    [0.884919	,0.898079	,0.828398	,0.871496	,0.923872], # seq ae
    [0.944006	,0.901434	,0.8586	,0.966706	,0.898342], # seq ae tempfreeze
    [0.881827	,0.937031	,0.8586	,0.92749	,0.901434], # seq single
])

# Names of the strategies
#strategy_names = [f"Strategy {i+1}" for i in range(data.shape[0])]
strategy_names = ['Image Only', 'Tabular Only', 'End-to-End', 'Hybrid AE', 'Hybrid Single', 'Sequential AE', 'Sequential AE Defreeze', 'Sequential Single']

# Initialize a matrix to store p-values
n_strategies = data.shape[0]
p_value_matrix = np.zeros((n_strategies, n_strategies))
results_matrix = np.full((n_strategies, n_strategies), "", dtype=object)

# Apply Bonferroni correction
alpha = 0.05
corrected_alpha = alpha / (n_strategies * (n_strategies - 1) / 2)

# Perform pairwise t-tests
for i, j in itertools.combinations(range(n_strategies), 2):
    # perform paired t-test
    stat, p = ttest_rel(data[i, :], data[j, :]) 
    p_value_matrix[i, j] = p
    p_value_matrix[j, i] = p  # Symmetric matrix
    
    # determine significance and direction
    mean_i = np.mean(data[i, :])
    mean_j = np.mean(data[j, :])
    
    if p < corrected_alpha:
        if mean_i > mean_j:
            results_matrix[i, j] = "✔ (better)"
            results_matrix[j, i] = "✘ (worse)"
        else:
            results_matrix[i, j] = "✘ (worse)"
            results_matrix[j, i] = "✔ (better)"
    else:
        results_matrix[i, j] = "Not Significant"
        results_matrix[j, i] = "Not Significant"

# Fill diagonal with "mean ± std" values
for i in range(n_strategies):
    mean = np.mean(data[i, :])
    std = np.std(data[i, :])
    results_matrix[i, i] = f"{mean:.3f} ± {std:.3f}"
    
# Convert to a Pandas DataFrame for better formatting
results_df = pd.DataFrame(results_matrix, index=strategy_names, columns=strategy_names)


In [3]:
# Style the matrix for better visualization
def highlight_cells(val, row, col):
    """Highlight cells based on significance and location."""
    #if row == col:  # Diagonal: mean ± std
    if "±" in val:
        return "background-color: lightblue; color: black; font-weight: bold; text-align: center;"
    elif "✔" in val:  # Better
        return "background-color: lightgreen; color: black; font-weight: bold; text-align: center;"
    elif "✘" in val:  # Worse
        return "background-color: lightcoral; color: black; font-weight: bold; text-align: center;"
    elif "Not Significant" in val:  # Not significant
        return "background-color: lightyellow; color: black; font-weight: bold; text-align: center;"
    else:
        return "text-align: center;"  # Default center alignment

# Apply the cell styles
styled_results = results_df.style.apply(
    lambda x: [highlight_cells(val, x.name, col) for col, val in enumerate(x)],
    axis=1
).set_caption(
    "AND Problem: BAcc Results with Statistical Testing"
).set_table_styles([
    {'selector': 'caption',
     'props': 'caption-side: top; font-size: 16px; font-weight: bold; text-align: center;'},
    {'selector': 'th',
     'props': 'text-align: center; font-weight: bold;'},
    {'selector': 'td',
     'props': 'text-align: center; font-weight: bold;'}
])

# Display the styled matrix
styled_results

,Image Only,Tabular Only,End-to-End,Hybrid AE,Hybrid Single,Sequential AE,Sequential AE Defreeze,Sequential Single
Image Only,0.607 ± 0.041,Not Significant,✘ (worse),✘ (worse),✘ (worse),✘ (worse),✘ (worse),✘ (worse)
Tabular Only,Not Significant,0.693 ± 0.034,✘ (worse),✘ (worse),✘ (worse),✘ (worse),✘ (worse),✘ (worse)
End-to-End,✔ (better),✔ (better),0.939 ± 0.029,Not Significant,Not Significant,✔ (better),Not Significant,Not Significant
Hybrid AE,✔ (better),✔ (better),Not Significant,0.925 ± 0.022,Not Significant,Not Significant,Not Significant,Not Significant
Hybrid Single,✔ (better),✔ (better),Not Significant,Not Significant,0.923 ± 0.014,Not Significant,Not Significant,Not Significant
Sequential AE,✔ (better),✔ (better),✘ (worse),Not Significant,Not Significant,0.881 ± 0.032,Not Significant,Not Significant
Sequential AE Defreeze,✔ (better),✔ (better),Not Significant,Not Significant,Not Significant,Not Significant,0.914 ± 0.038,Not Significant
Sequential Single,✔ (better),✔ (better),Not Significant,Not Significant,Not Significant,Not Significant,Not Significant,0.901 ± 0.029


In [7]:
# Create a significance matrix
significance_matrix = pd.DataFrame(
    np.where(p_value_matrix < corrected_alpha, "✔", "✘"),
    index=strategy_names,
    columns=strategy_names
)

# Fill diagonal with "—"
np.fill_diagonal(significance_matrix.values, "—")

# Style the matrix for better visualization
def highlight_significance(val):
    """Highlight cells based on significance."""
    if val == "✔":
        return "background-color: lightgreen; color: black; font-weight: bold;"
    elif val == "✘":
        return "background-color: lightcoral; color: black; font-weight: bold;"
    else:
        return ""  # No formatting for diagonal

styled_matrix = significance_matrix.style.applymap(highlight_significance).set_caption(
    "Melanoma Problem BAcc - Statistical Significance (Bonferroni Corrected)"
).set_table_styles([{
    'selector': 'caption',
    'props': 'caption-side: top; font-size: 16px; font-weight: bold; text-align: center;'
}])

# Display the styled matrix
styled_matrix

,Image Only,Tabular Only,End-to-End,Hybrid AE,Hybrid Single,Sequential AE,Sequential AE Defreeze,Sequential Single
Image Only,—,✘,✘,✘,✘,✘,✘,✔
Tabular Only,✘,—,✘,✘,✘,✘,✘,✘
End-to-End,✘,✘,—,✘,✘,✘,✘,✘
Hybrid AE,✘,✘,✘,—,✘,✘,✘,✘
Hybrid Single,✘,✘,✘,✘,—,✘,✘,✘
Sequential AE,✘,✘,✘,✘,✘,—,✘,✘
Sequential AE Defreeze,✘,✘,✘,✘,✘,✘,—,✘
Sequential Single,✔,✘,✘,✘,✘,✘,✘,—


In [4]:
import matplotlib.pyplot as plt
from matplotlib.table import Table
import numpy as np

# Function to create a table figure
def create_table_figure(df, filename="comparison_table.png"):
    fig, ax = plt.subplots(figsize=(len(df.columns) + 3, len(df) + 1))  # Adjust figure size based on table dimensions
    ax.axis('off')  # Turn off the axes

    # Create a table instance
    table = Table(ax, bbox=[0, 0, 1, 1])
    nrows, ncols = df.shape
    width, height = 1.0 / ncols, 1.0 / nrows

    # Add cells
    for (i, j), val in np.ndenumerate(df.values):
        if i == j:  # Diagonal: blue background
            cell = table.add_cell(
                i, j, width, height, text=val,
                facecolor="lightblue", loc="center", edgecolor="black"
            )
            # Set font size and boldness for diagonal cells
            cell.set_text_props(fontsize=16, fontweight="bold")
        elif "✔" in val:  # Better: green background
            cell = table.add_cell(
                i, j, width, height, text=val,
                facecolor="lightgreen", loc="center", edgecolor="black"
            )
            # Set font size for "✔"
            cell.set_text_props(fontsize=14)
        elif "✘" in val:  # Worse: red background
            cell = table.add_cell(
                i, j, width, height, text=val,
                facecolor="lightcoral", loc="center", edgecolor="black"
            )
            # Set font size for "✘"
            cell.set_text_props(fontsize=14)
        elif "Not Significant" in val:  # Not significant: yellow background
            cell = table.add_cell(
                i, j, width, height, text=val,
                facecolor="lightyellow", loc="center", edgecolor="black"
            )
            # Set font size for "Not Significant"
            cell.set_text_props(fontsize=14)

    # Add headers
    for j, col_name in enumerate(df.columns):
        cell = table.add_cell(
            -1, j, width, height, text=col_name,
            facecolor="lightgrey", loc="center", edgecolor="black"
        )
        # Set font size and boldness for headers
        cell.set_text_props(fontsize=16, fontweight="bold")
    for i, row_name in enumerate(df.index):
        cell = table.add_cell(
            i, -1, width, height, text=row_name,
            facecolor="lightgrey", loc="center", edgecolor="black"
        )
        # Set font size and boldness for row headers
        cell.set_text_props(fontsize=16, fontweight="bold")

    # Add the table to the axes
    ax.add_table(table)

    # Save the figure
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Table saved as {filename}")
    plt.close(fig)

# Example usage
create_table_figure(results_df, filename="comparison_table_larger_font.png")


Table saved as comparison_table_larger_font.png


# confirm normal distribution

In [2]:
# import loss data
data = [0.819	,0.808	,0.852	,0.792	,0.718]

stat, p = shapiro(data)
print(f"Shapiro-Wilk Test Statistic: {stat}, p-value: {p}")
if p > 0.05:
    print("Data looks normal (fail to reject H₀)")
else:
    print("Data does not look normal (reject H₀)")


Shapiro-Wilk Test Statistic: 0.9234218597412109, p-value: 0.5522507429122925
Data looks normal (fail to reject H₀)


In [9]:
# import auc data
data = [0.986577,	0.995789,	0.973418,	0.989604,	0.991315]

stat, p = shapiro(data)
print(f"Shapiro-Wilk Test Statistic: {stat}, p-value: {p}")
if p > 0.05:
    print("Data looks normal (fail to reject H₀)")
else:
    print("Data does not look normal (reject H₀)")


Shapiro-Wilk Test Statistic: 0.8942322731018066, p-value: 0.3788328170776367
Data looks normal (fail to reject H₀)


In [10]:
# import bacc data
data = [0.930846,	0.970325,	0.888275,	0.944269,	0.960258]

stat, p = shapiro(data)
print(f"Shapiro-Wilk Test Statistic: {stat}, p-value: {p}")
if p > 0.05:
    print("Data looks normal (fail to reject H₀)")
else:
    print("Data does not look normal (reject H₀)")


Shapiro-Wilk Test Statistic: 0.926429808139801, p-value: 0.5722349286079407
Data looks normal (fail to reject H₀)
